# 12_LLM_Fallback.ipynb
====================

This notebook demonstrates the **SupportAI Decision Engine**, a 3-tier customer support ticket routing architecture:
1. **High Confidence (Auto-Route)**: If intent classification confidence > high_threshold, return predicted category.
2. **Mid Confidence (LLM Fallback)**: If confidence is between low and high threshold, perform semantic search on historical resolved cases, construct a contextual instructions prompt, and query a cached small LLM (e.g., Phi-3 Mini) to generate a draft reply.
3. **Low Confidence (Human Escalation)**: If confidence < low_threshold, route directly to human customer support agents.

To support CPU execution on low-memory machines (>2GB RAM), the backend is configured to use a lightweight model.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if REPO_ROOT.name != "SupportAI" and (REPO_ROOT).exists():
    REPO_ROOT = REPO_ROOT
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
from pathlib import Path

from src.models.transformer.decision_engine import DecisionEngine
from src.utils.constants import OUTPUT_DIR

## 1. Define Engine Configuration
We specify the thresholds and model identifiers, defaulting to a lightweight, resource-friendly model for CPU inference.

In [2]:
config = {
    "decision_engine": {
        "high_confidence_threshold": 0.90,
        "low_confidence_threshold": 0.60
    },
    "llm": {
        "enabled": True,
        "provider": "huggingface",
        "model_id": "Qwen/Qwen2.5-0.5B-Instruct", # Or microsoft/Phi-3-mini-4k-instruct / Qwen/Qwen2.5-0.5B-Instruct
        "max_new_tokens": 64,
        "temperature": 0.2
    }
}

model_dir = OUTPUT_DIR / "models" / "best_model"
retriever_index_dir = OUTPUT_DIR / "retrieval_index"

print("Configuring Decision Engine...")

Configuring Decision Engine...


## 2. Initialise and Cache Decision Engine
We load all model components (intent classifier, FAISS index, SentenceTransformer, and small LLM) into memory once to minimise request latency.

In [3]:
engine = DecisionEngine(
    model_dir=model_dir,
    retriever_index_dir=retriever_index_dir,
    config=config
)
print("Decision Engine successfully loaded and cached in memory.")

[07/13/26 21:22:07] INFO     Loading intent classifier from:                                                       
                             C:\Users\gunav\Downloads\SupportAI\outputs\models\best_model

                    INFO     Loaded calibrated temperature scaling: T = 1.1939

                    INFO     Loading semantic retriever from:                                                      
                             C:\Users\gunav\Downloads\SupportAI\outputs\retrieval_index

                    INFO     Loading sentence embedding model: all-MiniLM-L6-v2

                    INFO     Load pretrained SentenceTransformer: all-MiniLM-L6-v2

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device set to use cuda:0


Decision Engine successfully loaded and cached in memory.


## 3. Tier 1: Auto Route (High Confidence)
We route a ticket where classifier confidence is high.

In [4]:
# Set threshold low to force auto-routing demonstration
engine.high_threshold = 0.01
res_high = engine.route_ticket("I forgot my passcode and cannot login")
print(json.dumps(res_high, indent=2))

{
  "intent": "passcode_forgotten",
  "confidence": 0.9707167744636536,
  "route": "classifier",
  "retrieved_docs": [],
  "llm_used": false,
  "reply": "Automated routing to category: passcode_forgotten"
}


## 4. Tier 2: LLM Fallback (Mid Confidence)
If confidence is within the medium range, the engine fetches top-3 similar resolved cases and drafts a reply using the LLM.

In [5]:
# Set high threshold high and low threshold low to trigger fallback demonstration
engine.high_threshold = 0.99
engine.low_threshold = 0.01
res_mid = engine.route_ticket("reset card pin number")
print(json.dumps(res_mid, indent=2))

{
  "intent": "get_physical_card",
  "confidence": 0.42254090309143066,
  "route": "fallback",
  "retrieved_docs": [
    {
      "rank": 1,
      "index": 1131,
      "score": 0.8320213556289673,
      "text": "how do i reset my pin, i can't seem to use my card?"
    },
    {
      "rank": 2,
      "index": 1284,
      "score": 0.7538939118385315,
      "text": "do you know how to change my card pin?"
    },
    {
      "rank": 3,
      "index": 735,
      "score": 0.7117278575897217,
      "text": "what do i do with my card pin?"
    }
  ],
  "llm_used": true,
  "reply": "I'm sorry, but I don't have enough information to provide a specific response for your request to reset your card pin number. Could you please provide me with more details about your situation?"
}


## 5. Tier 3: Human Escalation (Low Confidence)
If confidence is extremely low, the ticket is routed directly to a human specialist, saving LLM resource costs.

In [6]:
# Set thresholds high to force human escalation route
engine.high_threshold = 0.99
engine.low_threshold = 0.95
res_low = engine.route_ticket("random garbled input query")
print(json.dumps(res_low, indent=2))

{
  "intent": "unknown",
  "confidence": 0.20659328997135162,
  "route": "human_escalation",
  "retrieved_docs": [],
  "llm_used": false,
  "reply": "Escalated to human support review due to low confidence (0.2066)."
}


In [7]:
# Export Phase Manifest
from src.api.app import get_git_commit
from src.utils.artifacts import save_yaml

manifest = {
    "phase": "12_LLM_Fallback",
    "routing_decision_flow": "Decision engine built and validated with fallback",
    "git_commit": get_git_commit(),
}
save_yaml(manifest, REPO_ROOT / "outputs" / "manifests" / "phase_12_llm_fallback.yaml")
print("YAML manifest saved successfully:")
print(manifest)


[07/13/26 21:24:00] INFO     Saving YAML artifact to:                                                              
                             C:\Users\gunav\Downloads\SupportAI\outputs\manifests\phase_12_llm_fallback.yaml

YAML manifest saved successfully:
{'phase': '12_LLM_Fallback', 'routing_decision_flow': 'Decision engine built and validated with fallback', 'git_commit': 'ef9a0498221c5c43373fcf9e951987614174868f'}
